# 05. Boosting: HistGradientBoosting and LightGBM
**Run on:** Laptop B · **Approx. runtime:** ~4-6 min on a 4-core laptop


**Protocol (same for every model):**
- Each row is one *(GPS point, candidate arc)* pair. The label is 1 if the arc is the point's reference arc.
- **5-fold GroupKFold by trip** on the 78 usable training trips. Each fold fits on a 300k-row sample of points (KNN uses 100k) and is scored on all candidates of the held-out trips.
- The model is then **refit on the training split** and scored on the **10 student-test trips**.
- **Metrics:**
  - macro F1 / precision / recall: candidate level, threshold 0.5
  - log loss
  - **Top-1 accuracy**: the share of GPS points whose highest-scoring candidate is the reference arc
  - continuity
- The **HMM/Viterbi** decoder (parameters in `artifacts/hmm_params.json`) is then run on this model's probabilities to compare *classifier only* with *classifier + HMM*.


**Before running:** the dataset must be in `data/train/<id>/` and `data/student_test/<id>/`, and `pip install -r requirements.txt` done. Run all cells top to bottom (Kernel → Restart & Run All). Every number and plot below is computed when you run it.

In [ ]:
import os, sys, warnings
# run from notebooks/ or from the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
from pathsnap import nb
from pathsnap.models import make_model
from pathsnap.config import CFG
SEED = CFG["seed"]  # 42: every model and data split uses this seed
print("repo root:", ROOT)

In [ ]:
tr, tr_pts, te, te_pts = nb.load_data()

## HistGradientBoosting


**What it does:** gradient boosting. Trees are added **one after another**, each fitted to the errors (gradients of the log loss) of the current ensemble. Features are binned into histograms for speed.
- **Why:** boosting is usually the strongest family on tabular features like ours.
- **Limitation:** sequential training; more hyperparameters to tune.
- **Hyperparameters:** max_iter = 300, learning_rate = 0.1, max_leaf_nodes = 31, automatic early stopping.
- **Loss:** binary log loss.

In [ ]:
make_histgb = lambda: make_model("histgb", SEED)
print(make_histgb())
cv_histgb = nb.run_cv(make_histgb, tr, tr_pts, max_rows=300000)
cv_histgb.round(4)

In [ ]:
model_histgb, train_histgb, test_histgb, p_histgb = nb.fit_and_evaluate(make_histgb, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_histgb['continuity']:.4f} | inference {test_histgb['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("HistGradientBoosting", cv_histgb, train_histgb, test_histgb)

In [ ]:
imp_histgb = nb.plot_importance(model_histgb, te, "HistGradientBoosting")

In [ ]:
hmm_histgb = nb.hmm_compare(model_histgb, te, "HistGradientBoosting")
nb.plot_hmm(hmm_histgb, "HistGradientBoosting: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("histgb", cv_histgb, train_histgb, test_histgb,
                {"hmm_mean": hmm_histgb.drop(columns=["trip", "model"]).mean().to_dict()})

## LightGBM


**What it does:** gradient boosting with
- **histogram binning**: fast split search
- **leaf-wise growth**: always splits the leaf that reduces the loss most, so it learns fine interactions like *close AND same direction AND neighbours agree*
- row and feature subsampling plus L2 regularisation

Each new tree corrects the previous ones; the final score is the sum of all trees scaled by the learning rate.
- **Why it suits PathSnap:** handles mixed-unit features without scaling, captures interactions, gives well-calibrated probabilities (important for the HMM), and predicts very fast.
- **Limitation:** can overfit with large leaves; it is a black box apart from importances.
- **Hyperparameters (untuned):** n_estimators = 400, learning_rate = 0.05, num_leaves = 63, min_child_samples = 50, subsample = 0.8, colsample_bytree = 0.8, reg_lambda = 1.
- **Loss:** binary log loss.

In [ ]:
make_lightgbm = lambda: make_model("lightgbm", SEED)
print(make_lightgbm())
cv_lightgbm = nb.run_cv(make_lightgbm, tr, tr_pts, max_rows=300000)
cv_lightgbm.round(4)

In [ ]:
model_lightgbm, train_lightgbm, test_lightgbm, p_lightgbm = nb.fit_and_evaluate(make_lightgbm, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_lightgbm['continuity']:.4f} | inference {test_lightgbm['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("LightGBM", cv_lightgbm, train_lightgbm, test_lightgbm)

In [ ]:
imp_lightgbm = nb.plot_importance(model_lightgbm, te, "LightGBM")

In [ ]:
hmm_lightgbm = nb.hmm_compare(model_lightgbm, te, "LightGBM")
nb.plot_hmm(hmm_lightgbm, "LightGBM: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("lightgbm", cv_lightgbm, train_lightgbm, test_lightgbm,
                {"hmm_mean": hmm_lightgbm.drop(columns=["trip", "model"]).mean().to_dict()})

### LightGBM training curve
LightGBM records the loss after every boosting round. Here it is trained on 80% of the training trips and validated on the other 20% (a group split by trip) to show how train and validation log loss evolve.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from pathsnap.features import FEATURES
from pathsnap.dataset import subsample_points
i_tr, i_va = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(tr, groups=tr.trip))
fit = subsample_points(tr.iloc[i_tr], 300_000, SEED); va = tr.iloc[i_va]
curve_model = make_model("lightgbm", SEED)
curve_model.fit(fit[FEATURES].to_numpy(), fit.label.to_numpy(),
                eval_set=[(fit[FEATURES].to_numpy(), fit.label.to_numpy()), (va[FEATURES].to_numpy(), va.label.to_numpy())],
                eval_names=["train", "validation"], eval_metric="binary_logloss")
ev = curve_model.evals_result_
plt.figure(figsize=(7, 4))
plt.plot(ev["train"]["binary_logloss"], label="train"); plt.plot(ev["validation"]["binary_logloss"], label="validation (held-out trips)")
plt.xlabel("boosting round (tree)"); plt.ylabel("log loss"); plt.title("LightGBM training curve"); plt.legend(); nb.show()
best = int(np.argmin(ev["validation"]["binary_logloss"]))
print(f"lowest validation log loss {min(ev['validation']['binary_logloss']):.4f} at round {best + 1} of 400")

## Comparison of the models in this notebook

In [ ]:
nb.plot_cv_folds({"HistGradientBoosting": cv_histgb, "LightGBM": cv_lightgbm})
nb.plot_overfit({"HistGradientBoosting": (train_histgb, cv_histgb, test_histgb), "LightGBM": (train_lightgbm, cv_lightgbm, test_lightgbm)})
nb.plot_confusion({"HistGradientBoosting": test_histgb["confusion"], "LightGBM": test_lightgbm["confusion"]})
nb.plot_calibration(te.label.to_numpy(), {"HistGradientBoosting": p_histgb, "LightGBM": p_lightgbm})

In [ ]:
rows = []
for name, cv, t, h in [("HistGradientBoosting", cv_histgb, test_histgb, hmm_histgb), ("LightGBM", cv_lightgbm, test_lightgbm, hmm_lightgbm)]:
    rows.append({"model": name, "CV macro F1": cv.macro_f1.mean(), "CV std": cv.macro_f1.std(),
                 "test Top-1": t["top1"], "test macro F1": t["macro_f1"], "test log loss": t["log_loss"],
                 "test continuity": t["continuity"], "Top-1 + HMM (trip mean)": h.hmm_top1.mean(),
                 "route F1 + HMM (trip mean)": h.hmm_route_f1.mean()})
pd.DataFrame(rows).set_index("model").round(4)